In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os

# Paths to your COCO validation images & annotations in Drive
zip_val = "/content/drive/MyDrive/infosys_ai_vision_extract/val2017.zip"
zip_ann = "/content/drive/MyDrive/infosys_ai_vision_extract/annotations_trainval2017.zip"

# Extract validation images
with zipfile.ZipFile(zip_val, 'r') as zip_ref:
    zip_ref.extractall("/content")

# Extract annotations
with zipfile.ZipFile(zip_ann, 'r') as zip_ref:
    zip_ref.extractall("/content")

print("✅ COCO val images and annotations extracted")


In [ ]:
import cv2, glob, numpy as np, os
from pycocotools.coco import COCO
import albumentations as A

val_dir = "/content/val2017"
annFile = "/content/annotations/instances_val2017.json"
coco = COCO(annFile)

# Save folders
save_img_dir = "/content/augmented_dataset/images"
save_mask_dir = "/content/augmented_dataset/masks"
os.makedirs(save_img_dir, exist_ok=True)
os.makedirs(save_mask_dir, exist_ok=True)

resize_shape = (255,255)

# Augmentation pipeline
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=30, p=0.5)
])

val_images = sorted(glob.glob(val_dir+"/*.jpg"))

for idx, img_path in enumerate(val_images):
    img_id = int(os.path.basename(img_path).split(".")[0])
    I = cv2.imread(img_path)
    I = cv2.cvtColor(I, cv2.COLOR_BGR2RGB)

    ann_ids = coco.getAnnIds(imgIds=img_id, iscrowd=None)
    anns = coco.loadAnns(ann_ids)
    mask = np.zeros((I.shape[0], I.shape[1]), dtype=np.uint8)
    for ann in anns:
        m = coco.annToMask(ann)
        mask = np.maximum(mask, m)

    I_resized = cv2.resize(I, resize_shape)
    mask_resized = cv2.resize(mask, resize_shape, interpolation=cv2.INTER_NEAREST)

    augmented = transform(image=I_resized, mask=mask_resized)
    I_aug = augmented["image"]
    mask_aug = augmented["mask"]

    cv2.imwrite(f"{save_img_dir}/aug_{idx}.jpg", cv2.cvtColor(I_aug, cv2.COLOR_RGB2BGR))
    cv2.imwrite(f"{save_mask_dir}/aug_{idx}.png", mask_aug)

print("✅ Augmented dataset created: /content/augmented_dataset")


In [ ]:
# ======================================
# 1. Install segmentation_models_pytorch
# ======================================
!pip install -q segmentation-models-pytorch

# ======================================
# 2. Imports
# ======================================
import os, cv2, numpy as np, random
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import segmentation_models_pytorch as smp

In [ ]:
# ======================================
# 3. Dataset Class
# ======================================
class SegmentationDataset(Dataset):
    def __init__(self, img_dir, mask_dir, img_size=(128, 128)):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_files = sorted(os.listdir(img_dir))
        self.mask_files = sorted(os.listdir(mask_dir))
        self.img_size = img_size

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_files[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_files[idx])

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, self.img_size)
        img = img.astype(np.float32) / 255.0
        img = np.transpose(img, (2, 0, 1))
        img = torch.tensor(img, dtype=torch.float32)

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, self.img_size, interpolation=cv2.INTER_NEAREST)
        mask = (mask > 0).astype(np.float32)
        mask = np.expand_dims(mask, axis=0)
        mask = torch.tensor(mask, dtype=torch.float32)

        return img, mask


In [ ]:
# ======================================
# 4. Losses and Metrics
# ======================================
class DiceLoss(nn.Module):
    def forward(self, preds, targets, smooth=1e-6):
        preds = torch.sigmoid(preds)
        intersection = (preds * targets).sum()
        return 1 - (2 * intersection + smooth) / (preds.sum() + targets.sum() + smooth)

def combined_loss(preds, targets):
    return nn.BCEWithLogitsLoss()(preds, targets) + DiceLoss()(preds, targets)

def iou_pytorch(preds, targets, threshold=0.5):
    preds = (torch.sigmoid(preds) > threshold).float()
    intersection = (preds * targets).sum()
    union = preds.sum() + targets.sum() - intersection
    return (intersection / union).item() if union > 0 else 1.0


In [ ]:
# ======================================
# 5. Training Function with progress bars
# ======================================
def train_deeplab(lr, batch_size, num_epochs, subset_fraction=0.3, patience=3):
    dataset = SegmentationDataset("/content/augmented_dataset/images",
                                  "/content/augmented_dataset/masks")

    subset_size = int(len(dataset) * subset_fraction)
    subset_idx = random.sample(range(len(dataset)), subset_size)

    train_idx, val_idx = train_test_split(subset_idx, test_size=0.2, random_state=42)
    train_loader = DataLoader(Subset(dataset, train_idx), batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(Subset(dataset, val_idx), batch_size=batch_size, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = smp.DeepLabV3Plus(
        encoder_name="resnet34",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_iou = 0
    no_improve = 0

    for epoch in range(num_epochs):
        # ---------- Training ----------
        model.train()
        train_loss = 0
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        for imgs, masks in train_bar:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            preds = model(imgs)
            loss = combined_loss(preds, masks)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_bar.set_postfix(loss=train_loss/len(train_loader))

        # ---------- Validation ----------
        model.eval()
        val_iou = 0
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]  ")
        with torch.no_grad():
            for imgs, masks in val_bar:
                imgs, masks = imgs.to(device), masks.to(device)
                preds = model(imgs)
                val_iou += iou_pytorch(preds, masks)
                val_bar.set_postfix(val_iou=val_iou/len(val_loader))
        val_iou /= len(val_loader)
        print(f"Epoch {epoch+1}/{num_epochs} finished. Validation IoU: {val_iou:.4f}")

        # ---------- Early stopping ----------
        if val_iou > best_iou:
            best_iou = val_iou
            torch.save(model.state_dict(), "best_deeplab.pth")
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            print("Early stopping...")
            break

    return best_iou

# ======================================
# 6. Random Hyperparameter Search with progress
# ======================================
def random_hyperparam_search(n_trials=5):
    lr_choices = [1e-3, 5e-4, 1e-4]
    bs_choices = [4, 8]
    ep_choices = [10, 15]

    results = []

    for _ in range(n_trials):
        lr = random.choice(lr_choices)
        bs = random.choice(bs_choices)
        ep = random.choice(ep_choices)
        print(f"\nTesting LR={lr}, BS={bs}, Epochs={ep}")
        val_iou = train_deeplab(lr, bs, ep)
        results.append({"lr": lr, "batch_size": bs, "epochs": ep, "val_iou": val_iou})

    best_config = max(results, key=lambda x: x["val_iou"])
    print("\n✅ Best Hyperparameters:")
    print(best_config)

# Run random search
random_hyperparam_search(n_trials=5)

In [ ]:
# =============================
# Function to check Accuracy & IoU
# =============================
def pixel_accuracy(preds, masks):hnh
    preds = torch.sigmoid(preds) > 0.5   # convert logits → binary mask
    correct = (preds == masks.bool()).float()
    acc = correct.sum() / correct.numel()
    return acc.item()

def evaluate_model(model_path, dataset, batch_size=4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load model
    model = smp.DeepLabV3Plus(
        encoder_name="resnet34",
        encoder_weights=None,   # no pretrained weights when loading
        in_channels=3,
        classes=1
    ).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    # DataLoader
    test_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    total_iou, total_acc = 0, 0
    with torch.no_grad():
        for imgs, masks in tqdm(test_loader, desc="Testing"):
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs)

            total_iou += iou_pytorch(preds, masks)
            total_acc += pixel_accuracy(preds, masks)

    avg_iou = total_iou / len(test_loader)
    avg_acc = total_acc / len(test_loader)
    print(f"\n✅ Test IoU: {avg_iou:.4f}, Test Accuracy: {avg_acc:.4f}")
    return avg_iou, avg_acc


# =============================
# Example usage (after training)
# =============================
# Reload your dataset (or make a test split if you have one)
test_dataset = SegmentationDataset("/content/augmented_dataset/images",
                                   "/content/augmented_dataset/masks")

# Evaluate
evaluate_model("best_deeplab.pth", test_dataset, batch_size=4)
